# Erstelle Liste von Kreisen und Orten

Als Input dient [diese Seite](https://www.woerterbuchnetz.de/Woerterbuecher/WWB/KreiseOrte.html), die eine Liste aller Landschafts-, Kreis-, und Ortskürzel enthält. Sie ist mit HTML formatiert und dadurch sollten sich die Orte relativ einfach herausfiltern lassen.

In [17]:
import pandas as pd
import regex as re

In [18]:
# Open the raw HTML file
with open('../data/raw/KreiseOrte.html', 'r', encoding='latin-1') as f:
    html = f.read()

# Replace a few characters
html = re.sub(r'&#8594;', '→', html)
html = re.sub(r'\*', '', html)
html = re.sub(r'Ö¶', 'ö', html)
html = re.sub(r'Ö¼', 'ü', html)
html = re.sub(r'Ã¤', 'ä', html)
html = re.sub(r'Ã¶', 'ö', html)
html = re.sub(r'Ã¼', 'ü', html)
html = re.sub(r'Ã', 'Ö', html)
html = re.sub(r'Ã', 'Ü', html)

# First group ^= Landschaften, second group ^= Kreise und Orte
html = re.findall(r'<dl class="preface-definitionlist">(.+?)</dl>', html, flags=re.DOTALL)[1]

# Split up lines
html_lines = html.split('\n')

In [19]:
# Lists for holding the data that will later make up the dataframes
kreis_shorts = []
kreis_longs = []

ort_kreis = []
ort_shorts = []
ort_longs = []

for line in html_lines:
    # If the line contains a Kreis, remember all information about it
    new_kreis = re.findall(r'<dt class="preface-definitionterm-header">(.+?)</dt><dd class="preface-definition-header">(.+?)</dd>', line)
    if len(new_kreis):
        kreis_short = new_kreis[0][0]
        kreis_long = new_kreis[0][1]

        kreis_shorts.append(kreis_short)
        kreis_longs.append(kreis_long)

    # If the line contains an Ort, remember all information about it
    new_ort = re.findall(r'<dt class="preface-definitionterm"><a class="wwb-location-search-link" href="javascript:searchLocation\(\'WWB\',\'(?:.+?)\'\);">(.+?)</a></dt><dd class="preface-definition">(.+?)</dd>', line)
    if len(new_ort):
        ort_short = new_ort[0][0]
        ort_long = new_ort[0][1]

        ort_kreis.append(kreis_short)
        ort_shorts.append(ort_short)
        ort_longs.append(ort_long)

# Combine the lists created above into dataframes
kreis_df = pd.DataFrame({'Abkuerzung': kreis_shorts, 'Name': kreis_longs})
orte_df = pd.DataFrame({'Kreis': ort_kreis, 'Abkuerzung': ort_shorts, 'Name': ort_longs})

# Save the dataframes as CSVs
#kreis_df.to_csv('../data/processed/Kreise.csv', index=False, sep='\t')
#orte_df.to_csv('../data/processed/Orte.csv', index=False, sep='\t')

## Achtung! Nachdem die Liste der Orte automatisch erstellt wurde, müssen die großen Ö’s und Ü’s händisch eingefügt werden!

### Ergebnisse:

In [20]:
# Manuell geschriebene Tabelle
landschaften = pd.read_csv('../data/processed/Landschaften.csv', delimiter='\t')
landschaften

,Abkuerzung,Name,Beschreibung
0,Emsl,Emsland,"Emsland d. h. die Kreise Lingen, Meppen, Asche..."
1,HOsnabr,Hochstift Ösnabrück,"das frühere Hochstift Osnabrück, vornehmlich d..."
2,HPaderb,Hochstift Paderborn,"das frühere Hochstift Paderborn, d. h. die Kre..."
3,KSauerl,Kölnisches Sauerland,"Kölnisches Sauerland, d. h. die Kreise Arnsber..."
4,Lip,Lippe,"das frühere Land Lippe, d. h. die Kreise Detmo..."
5,Mark,Grafschaft Mark,die frühere Grafschaft Mark
6,Münsterl,Münsterland,Münsterland
7,Ravensbg,Grafschaft Ravensberg,die frühere Grafschaft Ravensberg
8,SOldbg,Südoldenburg,"Südoldenburg, d. h. die Kreise Vechta und Clop..."
9,WMünsterl,Westmünsterland,Westmünsterland


In [21]:
kreise = pd.read_csv('../data/processed/Kreise.csv', delimiter='\t')
kreise

,Abkuerzung,Name
0,Ahs,Kr. Ahaus
1,Alt,Kr. Altena u. die krfr. Stadt Lüdenscheid
2,Arn,Kr. Arnsberg
3,Asd,Kr. Aschendorf-Hümmling
4,Bbr,Kr. Bersenbrück
5,Bch,"die krfr. Städte Bochum, Herne u. Wattenscheid"
6,Bek,Kr. Beckum
7,Ben,Kr. Grafschaft Bentheim
8,Bie,Kr. Bielefeld
9,Bor,Kr. Borken u. die krfr. Stadt Bocholt


In [22]:
orte = pd.read_csv('../data/processed/Orte.csv', delimiter='\t')
orte

,Kreis,Abkuerzung,Name
0,Ahs,Ab,Asbeck
1,Ahs,Ae,Ahle (Heek)
2,Ahs,Ah,Ahaus
3,Ahs,Al,Alstätte
4,Ahs,Am,Ammeln
...,...,...,...
2727,Wol,We,Wettesingen
2728,Wol,Wh,Wenigenhasungen
2729,Wol,Wo,Wolfhagen
2730,Wtg,Lw,Langewiese
